In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "data").is_dir():
    project_root = project_root.parent
data_dir = project_root / "data"
figures_dir = project_root / "figures"
figures_dir.mkdir(exist_ok=True)

demo = pd.read_sas(data_dir / "DEMO_L.xpt", format="xport")
bmx = pd.read_sas(data_dir / "BMX_L.xpt", format="xport")
tchol = pd.read_sas(data_dir / "TCHOL_L.xpt", format="xport")
hdl = pd.read_sas(data_dir / "HDL_L.xpt", format="xport")
bpq = pd.read_sas(data_dir / "BPQ_L.xpt", format="xport")
smq = pd.read_sas(data_dir / "SMQ_L.xpt", format="xport")

for df in [demo, bmx, tchol, hdl, bpq, smq]:
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].apply(lambda x: np.where(np.abs(x) < 1e-50, 0.0, x))
    df["SEQN"] = df["SEQN"].astype(int)

df = (
    demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "INDFMPIR", "WTINT2YR", "WTMEC2YR", "SDMVPSU", "SDMVSTRA"]]
    .merge(bmx[["SEQN", "BMXBMI", "BMXWAIST", "BMXHT"]], on="SEQN", how="left")
    .merge(tchol[["SEQN", "LBXTC"]], on="SEQN", how="left")
    .merge(hdl[["SEQN", "LBDHDD"]], on="SEQN", how="left")
    .merge(bpq[["SEQN", "BPQ080", "BPQ101D"]], on="SEQN", how="left")
    .merge(smq[["SEQN", "SMQ020", "SMQ040"]], on="SEQN", how="left")
)

for col in ["BPQ080", "BPQ101D", "SMQ020", "SMQ040"]:
    df[col] = df[col].replace({7: np.nan, 9: np.nan, 77: np.nan, 99: np.nan})

flow = [("1_Initial", len(df))]

df = df[df["WTMEC2YR"] > 0]
flow.append(("2_MEC_Attended", len(df)))

df = df[df["RIDAGEYR"] >= 20]
flow.append(("3_Adults", len(df)))

df = df[df["RIDEXPRG"] != 1]
flow.append(("4_Not_Pregnant", len(df)))

df = df.dropna(subset=["BMXBMI", "BMXWAIST", "BMXHT"])
flow.append(("5_Valid_Anthro", len(df)))

df = df.dropna(subset=["LBXTC", "LBDHDD"])
flow.append(("6_Valid_Lipids", len(df)))

df = df.copy()

df["WHtR"] = df["BMXWAIST"] / df["BMXHT"]
df["Non_HDL_C"] = df["LBXTC"] - df["LBDHDD"]

df["BMI_Category"] = pd.cut(
    df["BMXBMI"],
    bins=[0, 18.5, 25.0, 30.0, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"],
    right=False
)
df["High_WHtR"] = (df["WHtR"] >= 0.50).astype(int)

print(pd.DataFrame(flow, columns=["Stage", "N_Participants"]))
print("\n")

vars_of_interest = ["Non_HDL_C", "WHtR", "BMXBMI", "RIDAGEYR", "INDFMPIR"]
desc_stats = df[vars_of_interest].agg(["mean", "std", "median"]).T
desc_stats["Q1"] = df[vars_of_interest].quantile(0.25)
desc_stats["Q3"] = df[vars_of_interest].quantile(0.75)
desc_stats["IQR"] = desc_stats["Q3"] - desc_stats["Q1"]
print(desc_stats)
print("\n")

mean_age_unweighted = df["RIDAGEYR"].mean()
mean_age_weighted = np.average(df["RIDAGEYR"], weights=df["WTMEC2YR"])
mean_lipid_unweighted = df["Non_HDL_C"].mean()
mean_lipid_weighted = np.average(df["Non_HDL_C"], weights=df["WTMEC2YR"])

print(f"Age: Unweighted={mean_age_unweighted:.2f}, Weighted={mean_age_weighted:.2f}")
print(f"Non-HDL-C: Unweighted={mean_lipid_unweighted:.2f}, Weighted={mean_lipid_weighted:.2f}")

sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
stages, counts = zip(*flow)
sns.barplot(x=list(counts), y=list(stages), color="steelblue")
plt.title("Sample Attrition Flow")
plt.tight_layout()
plt.savefig(figures_dir / "fig1_sample_flow.pdf")
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, var in enumerate(["Non_HDL_C", "BMXBMI", "WHtR"]):
    sns.histplot(df[var], kde=True, ax=axes[i], color="skyblue")
    axes[i].axvline(df[var].mean(), color='red', linestyle='dashed')
    axes[i].axvline(df[var].median(), color='black', linestyle='solid')
    axes[i].set_title(var)
plt.tight_layout()
plt.savefig(figures_dir / "fig2_distributions.pdf")
plt.close()

plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="BMI_Category", y="Non_HDL_C", hue="High_WHtR", palette="Set2")
plt.title("Non-HDL Cholesterol by BMI Category and Central Obesity Status")
plt.tight_layout()
plt.savefig(figures_dir / "fig3_group_comparisons.pdf")
plt.close()

            Stage  N_Participants
0       1_Initial           11933
1  2_MEC_Attended            8860
2        3_Adults            6064
3  4_Not_Pregnant            6023
4  5_Valid_Anthro            5709
5  6_Valid_Lipids            5205


                 mean        std      median          Q1          Q3  \
Non_HDL_C  134.025744  41.043435  130.000000  105.000000  158.000000   
WHtR         0.605959   0.100131    0.596457    0.532178    0.669385   
BMXBMI      29.692642   7.031666   28.400000   24.700000   33.400000   
RIDAGEYR    53.970413  16.910975   57.000000   40.000000   68.000000   
INDFMPIR     3.003641   1.642291    2.940000    1.520000    5.000000   

                 IQR  
Non_HDL_C  53.000000  
WHtR        0.137207  
BMXBMI      8.700000  
RIDAGEYR   28.000000  
INDFMPIR    3.480000  


Age: Unweighted=53.97, Weighted=48.95
Non-HDL-C: Unweighted=134.03, Weighted=135.13


In [6]:
from pathlib import Path

Path("figures").mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

# Figure 1
plt.figure(figsize=(10, 6))
stages, counts = zip(*flow)
sns.barplot(x=list(counts), y=list(stages), color="steelblue")
plt.title("Sample Attrition Flow")
plt.tight_layout()
plt.savefig("figures/fig1_sample_flow.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 2
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, var in enumerate(["Non_HDL_C", "BMXBMI", "WHtR"]):
    sns.histplot(df[var], kde=True, ax=axes[i], color="skyblue")
    axes[i].axvline(df[var].mean(), color="red", linestyle="dashed")
    axes[i].axvline(df[var].median(), color="black", linestyle="solid")
    axes[i].set_title(var)

plt.tight_layout()
plt.savefig("figures/fig2_distributions.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 3
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df,
    x="BMI_Category",
    y="Non_HDL_C",
    hue="High_WHtR",
    palette="Set2",
)
plt.title("Non-HDL Cholesterol by BMI Category and Central Obesity Status")
plt.tight_layout()
plt.savefig("figures/fig3_group_comparisons.png", dpi=300, bbox_inches="tight")
plt.close()